In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
import os
import cv2 as cv
import numpy as np

# Setting the common size of the images
SIZE = (224, 224)

images = []
labels = []

target_folder = './face_photos_lock_screen'

if not os.path.exists(target_folder):
    print("Target folder doesn't exist.")
    exit()

# Looping over the positive (entry) folder to load the images and assign labels
entry_folder = os.path.join(target_folder, 'entry')
for file_path in os.listdir(entry_folder):
    if file_path.endswith('.jpg'):
        image = cv.imread(os.path.join(entry_folder, file_path))
        image = cv.resize(image, SIZE)
        image = cv.normalize(image, None, alpha=0, beta=1, norm_type=cv.NORM_MINMAX, dtype=cv.CV_32F)
        images.append(image)
        labels.append(1)  # Assigning label 1 for positive (entry) images

# Looping over the negative (non-entry) folder to load the images and assign labels
non_entry_folder = os.path.join(target_folder, 'non_entry')
for file_path in os.listdir(non_entry_folder):
    if file_path.endswith('.PNG'):
        image = cv.imread(os.path.join(non_entry_folder, file_path))
        image = cv.resize(image, SIZE)
        image = cv.normalize(image, None, alpha=0, beta=1, norm_type=cv.NORM_MINMAX, dtype=cv.CV_32F)
        images.append(image)
        labels.append(0)  # Assigning label 0 for negative (non-entry) images

images = np.array(images)
labels = np.array(labels)

print('Shape of images:', images.shape)
print('Shape of labels:', labels.shape)


Shape of images: (877, 224, 224, 3)
Shape of labels: (877,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=22)

# the maximum dimension of X_train and X_test should be 2 so as to it can be given to a logistic regression model 
X_train = X_train.reshape(len(X_train), -1)
X_test = X_test.reshape(len(X_test), -1)

print('Shape of X_train:', X_train.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)


Shape of X_train: (701, 150528)
Shape of y_train: (701,)
Shape of X_test: (176, 150528)
Shape of y_test: (176,)


In [4]:
from sklearn.linear_model import LogisticRegression
import pickle

model = LogisticRegression(max_iter=800,random_state=22)
model.fit(X_train, y_train)

# Saving the trained model
with open('log_reg_face_detection_1.pkl', 'wb') as f:
    pickle.dump(model, f)


LogisticRegression(max_iter=800, random_state=22)

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
confusion_matrix = confusion_matrix(y_test, y_pred)
print('Accuracy:', accuracy)
print('Confusion Matrix:')
print(confusion_matrix)


Accuracy: 0.9829545454545454
Confusion Matrix:
[[137   0]
 [  3  36]]


In [7]:
%run face_detection.py
